In [1]:
import torch
from models.sam_image_encoder import ImageEncoderViT as SamImageEncoder

/home/clc/miniconda3/envs/var/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ckpt_path = "ckpt/sam_vit_b_01ec64.pth"

In [3]:
state_dict = torch.load(ckpt_path)

In [4]:
state_dict.keys()

dict_keys(['image_encoder.neck.0.weight', 'image_encoder.neck.1.weight', 'image_encoder.neck.1.bias', 'image_encoder.neck.2.weight', 'image_encoder.neck.3.weight', 'image_encoder.neck.3.bias', 'image_encoder.patch_embed.proj.weight', 'image_encoder.patch_embed.proj.bias', 'image_encoder.blocks.0.norm1.weight', 'image_encoder.blocks.0.norm1.bias', 'image_encoder.blocks.0.attn.rel_pos_h', 'image_encoder.blocks.0.attn.rel_pos_w', 'image_encoder.blocks.0.attn.qkv.weight', 'image_encoder.blocks.0.attn.qkv.bias', 'image_encoder.blocks.0.attn.proj.weight', 'image_encoder.blocks.0.attn.proj.bias', 'image_encoder.blocks.0.norm2.weight', 'image_encoder.blocks.0.norm2.bias', 'image_encoder.blocks.0.mlp.lin1.weight', 'image_encoder.blocks.0.mlp.lin1.bias', 'image_encoder.blocks.0.mlp.lin2.weight', 'image_encoder.blocks.0.mlp.lin2.bias', 'image_encoder.blocks.1.norm1.weight', 'image_encoder.blocks.1.norm1.bias', 'image_encoder.blocks.1.attn.rel_pos_h', 'image_encoder.blocks.1.attn.rel_pos_w', 'imag

In [5]:
image_encoder_state_dict = {}

for key, value in state_dict.items():
    if "image_encoder" in key:
        image_encoder_state_dict[key.replace("image_encoder.", "")] = value

In [6]:
image_encoder_state_dict.keys()

dict_keys(['neck.0.weight', 'neck.1.weight', 'neck.1.bias', 'neck.2.weight', 'neck.3.weight', 'neck.3.bias', 'patch_embed.proj.weight', 'patch_embed.proj.bias', 'blocks.0.norm1.weight', 'blocks.0.norm1.bias', 'blocks.0.attn.rel_pos_h', 'blocks.0.attn.rel_pos_w', 'blocks.0.attn.qkv.weight', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.proj.weight', 'blocks.0.attn.proj.bias', 'blocks.0.norm2.weight', 'blocks.0.norm2.bias', 'blocks.0.mlp.lin1.weight', 'blocks.0.mlp.lin1.bias', 'blocks.0.mlp.lin2.weight', 'blocks.0.mlp.lin2.bias', 'blocks.1.norm1.weight', 'blocks.1.norm1.bias', 'blocks.1.attn.rel_pos_h', 'blocks.1.attn.rel_pos_w', 'blocks.1.attn.qkv.weight', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.proj.weight', 'blocks.1.attn.proj.bias', 'blocks.1.norm2.weight', 'blocks.1.norm2.bias', 'blocks.1.mlp.lin1.weight', 'blocks.1.mlp.lin1.bias', 'blocks.1.mlp.lin2.weight', 'blocks.1.mlp.lin2.bias', 'blocks.2.norm1.weight', 'blocks.2.norm1.bias', 'blocks.2.attn.rel_pos_h', 'blocks.2.attn.rel_pos_w', '

In [7]:
from functools import partial


encoder_embed_dim=768
encoder_depth=12
encoder_num_heads=12
encoder_global_attn_indexes=[2, 5, 8, 11]

prompt_embed_dim = 256
image_size = 1024
vit_patch_size = 16

image_encoder=SamImageEncoder(
    depth=encoder_depth,
    embed_dim=encoder_embed_dim,
    img_size=image_size,
    mlp_ratio=4,
    norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
    num_heads=encoder_num_heads,
    patch_size=vit_patch_size,
    qkv_bias=True,
    use_rel_pos=True,
    global_attn_indexes=encoder_global_attn_indexes,
    window_size=14,
    out_chans=prompt_embed_dim,
)

In [8]:
image_encoder.load_state_dict(image_encoder_state_dict)

<All keys matched successfully>

In [9]:
image_encoder

ImageEncoderViT(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=True)
        (proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
      (mlp): MLPBlock(
        (lin1): Linear(in_features=768, out_features=3072, bias=True)
        (lin2): Linear(in_features=3072, out_features=768, bias=True)
        (act): GELU(approximate='none')
      )
    )
  )
  (neck): Sequential(
    (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (1): LayerNorm2d()
    (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (3): LayerNorm2d()
  )
)

In [10]:
prompt_encoder_state_dict = {}

for key, value in state_dict.items():
    if "prompt_encoder" in key:
        prompt_encoder_state_dict[key.replace("prompt_encoder.", "")] = value
prompt_encoder_state_dict.keys()

dict_keys(['pe_layer.positional_encoding_gaussian_matrix', 'point_embeddings.0.weight', 'point_embeddings.1.weight', 'point_embeddings.2.weight', 'point_embeddings.3.weight', 'not_a_point_embed.weight', 'mask_downscaling.0.weight', 'mask_downscaling.0.bias', 'mask_downscaling.1.weight', 'mask_downscaling.1.bias', 'mask_downscaling.3.weight', 'mask_downscaling.3.bias', 'mask_downscaling.4.weight', 'mask_downscaling.4.bias', 'mask_downscaling.6.weight', 'mask_downscaling.6.bias', 'no_mask_embed.weight'])

In [11]:
from models.prompt_encoder import PromptEncoder

In [12]:
image_embedding_size = 256 // 8

prompt_encoder=PromptEncoder(
    embed_dim=prompt_embed_dim,
    image_embedding_size=(image_embedding_size, image_embedding_size),
    input_image_size=(image_size, image_size),
    mask_in_chans=16,
)

In [13]:
prompt_encoder.load_state_dict(prompt_encoder_state_dict)

<All keys matched successfully>

In [14]:
dense_pe = prompt_encoder.get_dense_pe()
print(dense_pe.shape)

torch.Size([1, 256, 32, 32])
